# Train and run a single ACDC model

Train **one ResNet18**, with stacked ED/ES central SAX slices, random initialization, and seed 0. This uses the existing training pipeline and official patient splits. No architecture sweep is launched.

Use Python 3.10 or 3.11 with your separately installed **CineMA source checkout and its dependencies**. Install this repository's `requirements.txt` and Jupyter in that environment, then select it as the notebook kernel. CUDA is recommended; the pipeline otherwise uses CPU.

Run cells in order. First edit the processed-data path below. The default is a one-epoch smoke run; it exercises the full dataset and is not a meaningful trained result.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from IPython.display import display

# Find the repository when launched from its root or notebooks/ directory.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "main.py").is_file() and (p / "configs/acdc_2d.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the cnn-model-training checkout.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Repository:", PROJECT_ROOT)
print("Kernel:", sys.executable)

## 1. Choose data and training settings

`DATA_ROOT` must contain CineMA-preprocessed ACDC data, including `train_metadata.csv`, `test_metadata.csv`, and the corresponding patient NIfTI files. Raw ACDC images cannot be used directly.

If needed, preprocess first from a terminal in the project root:
```bash
python main.py --config-name acdc_2d 'run.stages=[preprocess]' data.acdc_raw=/path/to/raw/acdc data.acdc_processed=/path/to/processed/acdc
```

For full training set `SMOKE = False`. The example then uses 100 epochs maximum with early stopping; adjust this for your experiment. Smoke mode overrides epochs to 1, warmup to 0, effective batch size to 4, device batch size to 1, and workers to 0. Every model-selection list is explicitly restricted to one entry.

In [ ]:
DATA_ROOT = Path("/path/to/processed/acdc").expanduser().resolve()  # EDIT THIS
SMOKE = True
EPOCHS = 100
SEED = 0
BACKBONE = "resnet18"
FORMULATION = "stacked"
INITIALIZATION = "randinit"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "single_model_notebook"

# JSON quoting preserves paths containing spaces in Hydra overrides.
overrides = [
    f"data.acdc_processed={json.dumps(str(DATA_ROOT))}",
    f"logging.dir={json.dumps(str(OUTPUT_ROOT))}",
    f"model.models_2d=[{BACKBONE}]",
    f"model.formulations_2d=[{FORMULATION}]",
    f"model.initializations_2d=[{INITIALIZATION}]",
    f"run.seeds=[{SEED}]",
    f"run.smoke={str(SMOKE).lower()}",
    "run.resume=true",
    f"train.n_epochs={EPOCHS}",
    "train.n_warmup_epochs=0",
    "train.eval_interval=1",
    "train.patience=20",
    "train.batch_size=8",
    "train.batch_size_per_device=8",
    "train.n_workers=0",
]
base_command = [sys.executable, "-u", str(PROJECT_ROOT / "main.py"),
                "--config-name", "acdc_2d"]

def launch(extra):
    # Stream training output into the notebook using this kernel's environment.
    command = base_command + overrides + extra
    with subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

# Preview settings without loading patient data or training dependencies.
launch(["--cfg", "job"])

## 2. Validate the dataset

Validation checks the expected cohort, patient separation, and image inputs. Resolve any errors before training.

In [ ]:
for filename in ("train_metadata.csv", "test_metadata.csv"):
    if not (DATA_ROOT / filename).is_file():
        raise FileNotFoundError(f"Missing {DATA_ROOT / filename}; set DATA_ROOT or preprocess first.")
launch(["run.stages=[validate]"])

## 3. Train one model

The trainer selects the checkpoint with the best validation MCC and evaluates it on the test split after training. Keep the test split out of hyperparameter selection.

`run.resume=true` skips an already completed run when its summary exists; it does **not** resume optimizer state from an interrupted epoch. Use a new `OUTPUT_ROOT` for a fresh experiment. Smoke and full runs have separate artifact directories.

In [ ]:
launch(["run.stages=[validate,train]"])

## 4. Inspect results

Checkpoint metadata records the effective CineMA configuration. Training history, metrics, and predictions are stored in their respective output categories.

In [ ]:
import pandas as pd
from src.protocol import protocols
from src.utils import artifact_path

group = "architecture_smoke" if SMOKE else "architecture_bank"
run_dir = (OUTPUT_ROOT / "checkpoints" / group / "2d"
           / protocols["acdc_2d"]["task_key"] / BACKBONE / FORMULATION
           / INITIALIZATION / f"seed_{SEED}")
checkpoint_path = run_dir / "best_val_mcc.pt"
summary = json.loads((run_dir / "summary.json").read_text())
print("Checkpoint:", checkpoint_path)
print("Best epoch:", summary["best_epoch"])
print("Best validation MCC:", summary["best_val_mcc"])
display(pd.Series(summary["test"], name="Test metrics").to_frame())
display(pd.read_csv(artifact_path(run_dir / "history.csv")))
print("Patient predictions:", artifact_path(run_dir / "test" / "predictions.csv"))

## 5. Reload the model and predict one patient

Rebuild the model from checkpoint metadata and use the saved preprocessing configuration with the repository's evaluation transforms. The input is `[batch, 2, height, width]`, with ED then ES channels. This example predicts the first held-out test patient; class probabilities follow the checkpoint's class order.

In [ ]:
import torch
from omegaconf import OmegaConf
from src.models import ACDC2DClassifier
from src.dataset import acdc_2d_load_official_splits, acdc_2d_make_loaders

# Load the checkpoint produced by this notebook.
state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
architecture = json.loads((run_dir / "architecture.json").read_text())
inference_config = OmegaConf.create(architecture["config"])
inference_config.data.dir = str(DATA_ROOT)
inference_config.train.n_workers = 0
splits = acdc_2d_load_official_splits(DATA_ROOT)
_, _, test_loader = acdc_2d_make_loaders(DATA_ROOT, splits, inference_config, SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ACDC2DClassifier(state["backbone"], state["formulation"], state["initialization"])
model.load_state_dict(state["model"])
model = model.to(device).eval()
batch = next(iter(test_loader))
with torch.inference_mode():
    probabilities = model(batch["sax_image"].to(device)).softmax(dim=1)[0].cpu()
classes = list(state["classes"])
print("Patient:", batch["pid"][0])
print("Input shape:", tuple(batch["sax_image"].shape))
print("True class:", classes[int(batch["label"][0])])
print("Predicted class:", classes[int(probabilities.argmax())])
display(pd.DataFrame({"class": classes, "probability": probabilities.numpy()}))